# Timestral-209M (zero-shot) + LightGBM (ensemble)

Modèle pré-entraîné de Mistral : pas d'entraînement. Pour chaque arc, il lit les 2048 dernières heures avant `t0`, ainsi que les variables de calendrier connues jusqu'à la fin de l'horizon, plus un encodage cyclique heure/jour (sin/cos). La prédiction couvre le gap d'un jour et les 120 heures évaluées, comme dans `chronos.ipynb`.

**Résultats du backtest (38 dates, focus Convention + Saints-Pères, RMSE mean/date) :**
- Timestral seul : 64.3 | LightGBM seul : 67.2 | naif lag7j : 92.8
- contexte 1024h/4096h pire que 2048h ; covariables supplémentaires (profils, lags) pires
- encodage cyclique heure/jour : aide Champs-Élysées, neutre sur les 2 autres arcs
- **ensemble Timestral + ~35% LightGBM : 62.0** ← optimum large et robuste (w=0.25 à 0.5 tous ≈ 62)

Timestral s'installe depuis le dépôt officiel (Python >= 3.10, PyTorch >= 2.4). Si nécessaire, exécuter la cellule d'installation une fois, puis redémarrer le kernel.

In [ ]:
%pip install "git+https://github.com/mistralai/timestral.git"

In [ ]:
import sys; sys.path.append("..")  # pour importer dataset_create depuis notebooks/
import numpy as np
import pandas as pd
from dataset_create import charger, split, ORIGINES, HORIZON, feature_engineering

df = charger(imputer=True)  # imputer=False : trous laissés en NaN
rmse = lambda y, p, axe: ((y - p) ** 2).groupby(axe, observed=True).mean() ** .5  # RMSE par arc

In [ ]:
import torch
import lightgbm as lgb
from timestral import TimestralForecaster
from dataset_create import ajouter_futur, vraies_valeurs

pipe = TimestralForecaster.from_pretrained(
    "mistralai/Timestral-209M",
    device="cuda" if torch.cuda.is_available() else "cpu",
)
fe, _ = ajouter_futur(df, "debit", vraies_valeurs(df).datetime.max())  # même index que split() et test_final()

# Covariables connues : calendrier + encodage cyclique heure/jour (sin/cos)
COV_NUM = ["ferie", "pont", "paris_respire", "evenement", "confinement",
           "heure_sin", "heure_cos", "jour_sin", "jour_cos"]
COV_CAT = ["vacances_code", "type_jour"]
CTX = 2048  # 2048h optimal (testé 1024/2048/4096)
W_LGB = 0.35  # poids LightGBM dans l'ensemble (38 dates : optimum plat 0.25-0.5, 0.35 au milieu)

# Timestral attend des covariables connues sur le contexte ET tout l'horizon.
vacances_code = fe.vacances.cat.codes.astype(float).where(fe.vacances.cat.codes >= 0)
g = fe.assign(
    item_id=fe.axe.astype(str),
    timestamp=fe.datetime.dt.tz_convert(None),  # UTC sans fuseau : grille horaire régulière
    vacances_code=vacances_code,
    type_jour=fe.type_jour.astype(float),
    heure_sin=np.sin(2 * np.pi * fe.datetime.dt.hour / 24),
    heure_cos=np.cos(2 * np.pi * fe.datetime.dt.hour / 24),
    jour_sin=np.sin(2 * np.pi * fe.datetime.dt.dayofweek / 7),
    jour_cos=np.cos(2 * np.pi * fe.datetime.dt.dayofweek / 7),
).astype({c: np.float32 for c in COV_NUM + COV_CAT})

# LightGBM : params optimaux trouvés dans lightgbm.ipynb (RMSE ~90 seul, utile en ensemble)
LGB_PARAMS = {"debit": dict(n_estimators=1000, learning_rate=0.05, num_leaves=127, objective="huber", alpha=100),
              "occupation": dict(n_estimators=500, learning_rate=0.05, num_leaves=31, min_child_samples=20)}

def modele(X_train, y_train, X_test, cible="debit"):
    t0 = fe.loc[X_train.index, "datetime"].max().tz_convert(None) + pd.Timedelta("1h")
    H = int((fe.loc[X_test.index, "datetime"].max().tz_convert(None) - t0) / pd.Timedelta("1h")) + 1  # de t0 à la dernière heure testée
    debut = t0 - pd.Timedelta(hours=CTX)
    heures = pd.date_range(debut, periods=CTX + H, freq="h")
    predictions = []

    for axe in X_test.axe.astype(str).unique():
        serie = g.loc[g.item_id == axe].set_index("timestamp").reindex(heures)
        contexte = serie[cible].to_numpy(dtype=np.float32)[:CTX][None, None, :]
        connues = serie[COV_NUM].to_numpy(dtype=np.float32).T[None, :, :]
        categories = serie[COV_CAT].to_numpy(dtype=np.float32).T[None, :, :]

        forecast = pipe.forecast(
            contexte,
            prediction_length=H,
            known_covariates=connues,
            categorical_known_covariates=categories,
        )
        mediane = forecast.median[0, 0].cpu().numpy()
        predictions.append(pd.DataFrame({
            "item_id": axe,
            "timestamp": heures[CTX:],
            "prediction": mediane,
        }))

    pred = pd.concat(predictions, ignore_index=True)
    test = pd.DataFrame({
        "item_id": X_test.axe.astype(str),
        "timestamp": fe.loc[X_test.index, "datetime"].dt.tz_convert(None),
    })
    p_timestral = test.merge(pred, on=["item_id", "timestamp"], how="left", validate="many_to_one").prediction.to_numpy()

    # Ensemble avec LightGBM (poids W_LGB)
    p_lgb = lgb.LGBMRegressor(verbose=-1, random_state=0, **LGB_PARAMS[cible]).fit(X_train, y_train).predict(X_test).clip(0)
    return W_LGB * p_lgb + (1 - W_LGB) * p_timestral

In [ ]:
# 1 semaine : dernière donnée le ven. 07/11/2025 -> on prédit sam. 08 -> mer. 12/11 (11/11 férié inclus)
CIBLE = "debit"  # ou "occupation"
X_train, X_test, y_train, y_test = split(df, "2025-11-07", cible=CIBLE)
p = modele(X_train, y_train, X_test, CIBLE)
print(pd.DataFrame({"modele": rmse(y_test, p, X_test.axe), "naif_lag7j": rmse(y_test, X_test.lag_7j, X_test.axe)}).round(1))

In [ ]:
# Les 3 tests communs, contre le naïf lag_7j : semaine de Lucie (07/11/2025), test final, backtest sur TOUS les vendredis
from evaluation import trois_tests
P = trois_tests(df, lambda X_train, y_train, X_test: modele(X_train, y_train, X_test, CIBLE), "timestral_lgb", CIBLE)  # Timestral + 35 % LightGBM